# Shin et al. (2025) from dandiset 000248: checks of the decoder and other decoder types

Dandiset [000248](https://dandiarchive.org/dandiset/000248/0.240502.2344) version 0.240502.2344.
This notebook repeats decoder reading (a) of `decoding_shin_2025.ipynb` with other cross-validation
seeds (for fig3c-npx-v1-inference and fig3d-npx-areas-inference; reading (b) with the same seeds is in
`decoding_seed1_shin_2025.ipynb` and `decoding_seed2_shin_2025.ipynb`), checks the bound of the C
search and the coding of decoder reading (b) in V1, PM and AM, and tests supp3-decoder-types.

| Claim | Tested here |
|---|---|
| fig3c-npx-v1-inference, fig3d-npx-areas-inference | seeds of reading (a) only; outcomes in `decoding_shin_2025.ipynb` |
| supp3-decoder-types | quadratic and RBF SVMs; the network is timed only |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import warnings

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import stats

import illusion as il

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
N_JOBS = 4

started = time.time()
sessions = il.load_all(ccg=False)
print(f"loaded {len(sessions)} sessions of dandiset {il.DANDISET_ID} version {il.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

RS units per area and session, and ICwcfg1 trials of the decoded images.

In [ ]:
rows = []
for s in sessions:
    u, b = s["units"], s["blocks"]["ICwcfg1"]
    n = {c: int(np.sum(b["codes"] == c)) for c in [il.IC1, il.LC1, il.LC2, il.IC2, il.TRE1, il.TRE2, il.XRE1, il.XRE2]}
    rows.append({"subject": s["subject"], **{f"{a}_RS": int(((u.area == a) & u.rs).sum()) for a in il.AREAS},
                 "trials IC1/LC1/LC2/IC2": "/".join(str(n[c]) for c in [il.IC1, il.LC1, il.LC2, il.IC2]),
                 "TRE1/TRE2": f"{n[il.TRE1]}/{n[il.TRE2]}", "XRE1/XRE2": f"{n[il.XRE1]}/{n[il.XRE2]}"})
units_tab = pd.DataFrame(rows)
print(units_tab.to_string(index=False))
allR = np.concatenate([s["blocks"]["ICwcfg1"]["R"].ravel() for s in sessions])
print(f"IC response (spikes/s): min {allR.min():.1f}, median {np.median(allR):.1f}, max {allR.max():.1f}")
print("Session-areas with >= 50 RS units:", {a: int((units_tab[f"{a}_RS"] >= 50).sum()) for a in il.AREAS},
      "(paper 12, 8, 11, 10, 12, 11)")

The counts are those of the data check of `decoding_shin_2025.ipynb`.

## Cross-validation seeds (fig3c, fig3d)

The tenfold partition is random. Reading (a) is repeated on every area with two more seeds (the
seed of `decoding_shin_2025.ipynb` plus 100,000 and 200,000), the TRE decoder only. Reading (b)
with the same seeds, on every area, is in `decoding_seed1_shin_2025.ipynb` and
`decoding_seed2_shin_2025.ipynb`. The values of the first seed are those of the decoding notebook.

In [ ]:
t0 = time.time()
jobs = []
for i, s in enumerate(sessions):
    u = s["units"]
    for j, a in enumerate(il.AREAS):
        m = ((u.area == a) & u.rs).values
        if m.sum() < 50:
            continue
        b = s["blocks"]["ICwcfg1"]
        jobs.append((s["subject"], a, b["R"][:, m], b["codes"], 1000 * i + j))

def tre(subject, a, R, codes, seed, rd, kernel="linear"):
    r = il.decode(R, codes, [il.IC1, il.LC1, il.LC2, il.IC2], [il.TRE1, il.TRE2], seed=seed, reading=rd, kernel=kernel)
    score, icv, lcv = il.inference_score(r["probe"])
    return {"subject": subject, "area": a, "seed": seed, "reading": rd, "kernel": kernel, "acc": r["acc"],
            "tre_ic": icv, "tre_lc": lcv, "inference": score, "unconverged": r["n_unconverged"]}

tasks = [(sub, a, R, c, sd + k * 100000, "a") for k in [1, 2] for (sub, a, R, c, sd) in jobs]
seeds = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(tre)(*t) for t in tasks))
seeds["seed_k"] = seeds.seed // 100000
print(f"{len(tasks)} decoders in {time.time() - t0:.0f} s; fits that did not converge: {int(seeds.unconverged.sum())}")

In [ ]:
for (rd, k), x in seeds[seeds.area == "V1"].groupby(["reading", "seed_k"]):  # reading (a); (b) in decoding_seed1 and seed2
    print(f"V1, reading ({rd}), seed +{k}00000: P(TRE -> IC) {x.tre_ic.mean():.3f}, P(TRE -> LC) {x.tre_lc.mean():.3f}, "
          f"accuracy {x.acc.mean():.3f}; signed-rank IC vs LC p = {il.signrank_p(x.tre_ic, x.tre_lc):.4g}; "
          f"IC within 0.51 +/- 0.051: {abs(x.tre_ic.mean() - 0.51) <= 0.051}, LC within 0.22 +/- 0.022: {abs(x.tre_lc.mean() - 0.22) <= 0.022}")

In [ ]:
rows = []
for k, x in seeds[seeds.reading == "a"].groupby("seed_k"):
    for a in il.AREAS:
        y = x[x.area == a]
        rows.append({"seed_k": k, "area": a, "sessions": len(y), "inference_mean": y.inference.mean(), "p": il.signrank_p(y.inference)})
t = pd.DataFrame(rows)
print(t.pivot(index="area", columns="seed_k", values=["inference_mean", "p"]).reindex(il.AREAS).round(4).to_string())
for k, y in t.groupby("seed_k"):
    sig = y[(y.p < 0.05) & (y.inference_mean > 0)].area.tolist()
    print(f"reading (a), seed +{k}00000: significant {sig} ({len(sig)} areas); three largest mean scores "
          f"{y.nlargest(3, 'inference_mean').area.tolist()}; three smallest p {y.nsmallest(3, 'p').area.tolist()}")

**Result:** in V1, under reading (a), both TRE values are within tolerance with each further seed (IC 0.513 and 0.511, LC 0.198 and 0.199), with signed-rank p = 0.0024 for both. Each further seed gives the same five significant areas as the decoding notebook (V1, LM, RL, AL, AM; PM p = 0.092) and the same three largest mean scores (V1, LM, AL). Under reading (a), the fig3c and fig3d results do not depend on the seed; reading (b) is in the two seed notebooks.

## Bound of the C search and coding (decoder reading b)

MATLAB's `OptimizeHyperparameters` for `fitcecoc` searches the coding (one-vs-one or one-vs-all)
and the SVM's box constraint and kernel scale over 10^-3 to 10^3. For a linear kernel, the kernel
scale s divides the inputs, which gives the same classifier as C / s^2, so a search over C covers
both. Reading (b) searches C over 10^-3 to 1 with one-vs-one coding. Here, on the training trials of
the first outer fold of each session, in V1 and in PM and AM (the two areas whose significance in
fig3d depends on the decoder), the inner threefold accuracy is computed for C from 10^-3 to 10^3
(one-vs-one) and for C up to 1 with one-vs-all coding, to show whether the bound and the coding can
change the choice.

In [ ]:
t0 = time.time()
path_jobs = [j for j in jobs if j[1] in ("V1", "PM", "AM")]
cp = Parallel(n_jobs=N_JOBS)(delayed(il.c_path)(R, c, [il.IC1, il.LC1, il.LC2, il.IC2], sd) for (sub, a, R, c, sd) in path_jobs)
cp = pd.concat([d.assign(subject=sub, area=a) for d, (sub, a, *_ ) in zip(cp, path_jobs)])
print(f"computed in {time.time() - t0:.0f} s")
for a in ["V1", "PM", "AM"]:
    x = cp[cp.area == a]
    print(f"\n{a}, inner threefold accuracy by coding and C:")
    print(x.pivot_table(index="subject", columns=["coding", "C"], values="inner_cv_acc").round(3).to_string())
    best_le1 = x[x.C <= 1].groupby("subject").inner_cv_acc.max()
    best_gt1 = x[x.C > 1].groupby("subject").inner_cv_acc.max()
    best_ovo = x[(x.C <= 1) & (x.coding == "ovo")].groupby("subject").inner_cv_acc.max()
    best_ovr = x[(x.C <= 1) & (x.coding == "ovr")].groupby("subject").inner_cv_acc.max()
    d = best_gt1 - best_le1
    print(f"{a}: best with C > 1 minus best with C <= 1 (both codings), per session:", d.round(3).to_dict())
    print(f"{a}: sessions where C > 1 is better: {int((d > 0).sum())} of {len(d)}; range {d.min():.3f} to {d.max():.3f}")
    e = best_ovr - best_ovo
    print(f"{a}: best one-vs-all minus best one-vs-one, C <= 1: range {e.min():.3f} to {e.max():.3f}")
    print(f"{a}: one-vs-one fits that did not converge, by C (of {3 * x.subject.nunique()} per C):",
          x[x.coding == "ovo"].groupby("C").unconverged.sum().to_dict())

**Result:** the best inner accuracy with C above 1 is lower than the best with C up to 1 in every V1 and AM session (by 0.005 to 0.064 in V1 and 0.005 to 0.088 in AM) and in 11 of 12 PM sessions (by 0.001 to 0.113); in PM session 637484 it is higher by 0.001. Most fits with C above 1 stop at max_iter (12 to 17 of 36 per C value in V1, 28 to 30 of 36 in PM, 30 of 33 in AM), so these accuracies come partly from unconverged fits, and the check covers the first outer fold only. On these fits, extending the grid above C = 1 would change the chosen C in at most one session of the three areas, by 0.001 in inner accuracy. The one-vs-all coding changes the best inner accuracy by -0.012 to 0.024 in V1, -0.024 to 0.017 in PM and -0.026 to 0.027 in AM, so it could change the choice in some sessions; its effect on the inference scores was not computed.

## supp3-decoder-types

> Decoders of other kinds (quadratic and radial basis function SVMs, a fully connected neural
> network) give results highly consistent with the linear SVM.

The quadratic and RBF SVMs replace the linear kernel in decoder reading (a) (C = 1), with the
first extra seed of the section above, on V1, LM and AL; the paper gives no kernel parameters, so
scikit-learn's data-scaled `gamma="scale"` is assumed (helper, `decode`). Each is compared with
the linear SVM on the same seed: sign of the mean TRE inference score and significance of the
signed-rank test against 0 (p < 0.05). The network of the Methods (two hidden layers of 100 units,
softmax output, Adam, cross-entropy, 50,000 epochs, tenfold) is timed on one fold of one session,
and its full cost estimated.

In [ ]:
t0 = time.time()
tasks = [(sub, a, R, c, sd + 100000, "a", k) for k in ["poly", "rbf"] for (sub, a, R, c, sd) in jobs if a in ("V1", "LM", "AL")]
kern = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(tre)(*t) for t in tasks))
kern = pd.concat([kern, seeds[(seeds.reading == "a") & (seeds.seed_k == 1) & seeds.area.isin(["V1", "LM", "AL"])]])
print(f"{len(tasks)} decoders in {time.time() - t0:.0f} s")
rows = []
for (a, k), x in kern.groupby(["area", "kernel"]):
    rows.append({"area": a, "kernel": k, "sessions": len(x), "accuracy": x.acc.mean(), "IC": x.tre_ic.mean(), "LC": x.tre_lc.mean(),
                 "inference_mean": x.inference.mean(), "p": il.signrank_p(x.inference)})
kt = pd.DataFrame(rows)
kt["sig_positive"] = (kt.p < 0.05) & (kt.inference_mean > 0)
print(kt.round(4).to_string(index=False))
lin = kt[kt.kernel == "linear"].set_index("area")
for k in ["poly", "rbf"]:
    y = kt[kt.kernel == k].set_index("area")
    agree = {a: bool((np.sign(y.loc[a, "inference_mean"]) == np.sign(lin.loc[a, "inference_mean"])) and
                     (y.loc[a, "sig_positive"] == lin.loc[a, "sig_positive"])) for a in ["V1", "LM", "AL"]}
    print(f"{k}: same sign and significance as linear in", agree)

In [ ]:
from sklearn.neural_network import MLPClassifier
s = sessions[0]
b = s["blocks"]["ICwcfg1"]
m = ((s["units"].area == "V1") & s["units"].rs).values
sel = np.isin(b["codes"], [il.IC1, il.LC1, il.LC2, il.IC2])
X = stats.zscore(b["R"][sel][:, m], axis=0)
X[~np.isfinite(X)] = 0
y = b["codes"][sel]
n_ep = 300
net = MLPClassifier(hidden_layer_sizes=(100, 100), solver="adam", batch_size=len(y), max_iter=n_ep, tol=0,
                    n_iter_no_change=10 ** 9, alpha=0.0, random_state=0)
t0 = time.time()
net.fit(X[: int(0.9 * len(y))], y[: int(0.9 * len(y))])
per_epoch = (time.time() - t0) / n_ep
n_fits = 10 * len([j for j in jobs if j[1] in ("V1", "LM", "AL")])
print(f"{per_epoch * 1000:.2f} ms per full-batch epoch on {m.sum()} units; one fit of 50,000 epochs ~ {per_epoch * 5e4 / 60:.1f} min; "
      f"{n_fits} fits (10 folds x {n_fits // 10} session-areas of V1, LM, AL) ~ {per_epoch * 5e4 * n_fits / 3600:.1f} h on one core")

**Result: blocked (too_large) for the network; the two SVMs agree with the linear SVM.** With the quadratic and RBF kernels, the TRE inference score is positive and significant in V1, LM and AL, as with the linear SVM on the same seed (p from 0.0015 to 0.0156 for all three kernels). The network of the Methods (50,000 full-batch epochs, tenfold) takes 4.4 ms per epoch in this run, about 3.7 minutes per fit and about 18 hours on one core for the 300 fits of these three areas (the timing varies between runs), far beyond the notebook's limit, so the claim, which needs each decoder type, is not decided.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")

## Summary

| Check or claim | Result |
|---|---|
| Seeds for fig3c and fig3d, reading (a) | fig3c within tolerance and fig3d five significant areas with each seed; reading (b) in the seed notebooks |
| Bound of the C search and coding (V1, PM, AM) | C above 1 better in 1 of 35 sessions, by 0.001, from mostly unconverged fits; one-vs-all changes the best inner accuracy by -0.026 to 0.027 |
| supp3-decoder-types | blocked (too_large) for the network; quadratic and RBF SVMs agree with the linear SVM in V1, LM and AL |

How the analysis differs from the paper's: the quadratic and RBF kernels use scikit-learn's
data-scaled `gamma="scale"`, as the paper gives no kernel parameters; the network is timed with
scikit-learn's MLPClassifier, where the paper used PyTorch.